<h1 align=center style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
GTA VI
</font>
</h1>


In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import log_loss, mean_squared_log_error
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier, CatBoostRegressor
pd.set_option('display.max_columns', None)
print('All imports successful')

<h2 align=left style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
The Dataset
</font>
</h2>


<center>
<div dir="ltr" style="line-height:200%;font-family:vazir;font-size:medium">
<font face="vazir" size="3">

| **Column Name** | **Description** |
| --- | --- |
| `transaction_id` | Unique identifier for each record in the dataset |
| `year` | Calendar year of the record |
| `month` | Calendar month of the record (1 to 12) |
| `quarter` | Financial quarter derived from the month (1 to 4) |
| `country` | Name of the country where the sale was made |
| `iso3_code` | Three-letter ISO code (ISO 3166-1 alpha-3) for the country |
| `region` | Continent or geographical region of the country |
| `platform` | Gaming platform (e.g., PS3, PS4, PC, Xbox Series X\|S) |
| `game_edition` | Purchased edition of the game (Standard, Premium, Legacy, etc.) |
| `units_sold` | Number of copies sold in that month/country/platform |
| `new_customers` | Number of buyers who purchased for the first time in that period |
| `returning_customers` | Number of returning buyers (those who have purchased before) |
| `estimated_active_players` | Estimated number of unique active players |
| `peak_concurrent_players` | Highest number of concurrent players recorded in that month |
| `online_players` | Number of active players in any of the online sections |
| `story_mode_players` | Number of players engaged with the story mode content (single-player) |
| `gta_online_players` | Number of active players specifically in the GTA Online section |
| `average_playtime_hours` | Average playtime hours per active player in that month |
| `average_session_length_minutes` | Average length of each gaming session in minutes |
| `holiday_season` | Was the sale during the holiday season (November, December, January)? (0 or 1) |
| `major_sale_event` | Name of the special sale event active in that month (if any) |
| `marketing_campaign` | Was a marketing campaign active during that period? (0 or 1) |
| `customer_rating` | Average rating submitted by users (from 4.2 to 5.0) |
| `review_count` | Number of submitted reviews |
| `refund_rate_percentage` | Percentage of refunded purchases |
| `currency` | Local currency code of the target country |
| `exchange_rate_to_usd` | Local currency to USD exchange rate at the time of sale |
| `internet_penetration_percentage` | Internet penetration percentage in the target country |
| `gaming_market_size` | Estimated relative size of the gaming market in the country |
| `population_millions` | Population of the target country in millions |
| `gdp_per_capita_usd` | GDP per capita of the country in USD (indicating purchasing power) |
| `release_phase` | Platform release life cycle phase (Launch, Growth, Mature, Legacy) |
| `weekend_sales_percentage` | Percentage share of sales on weekends |
| `weekday_sales_percentage` | Percentage share of sales on weekdays |
| `season` | Meteorological season (Winter, Spring, Summer, Autumn) |
| `special_event` | Special event that stimulated sales (e.g., release of a major update) |
| `top_game_category` | Main genre category of the game (Action-Adventure) |
| `platform_generation` | Gaming console generation (e.g., Gen7, Gen8, Gen9, PC) |
| `gross_revenue_usd` | **(First Target Column - Regression):** Gross revenue earned in USD |
| `sales_channel` | **(Second Target Column - Classification):** Sales channel, with values `Physical` or `Digital` |

</font>
</div>
</center>

In [ ]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
print(f'Train shape: {train.shape}')
print(f'Test shape: {test.shape}')

target_col = 'gross_revenue_usd'
target_cls = 'sales_channel'

train['is_revenue_missing'] = (train[target_col] == 'Lost').astype(int)
train[target_col] = train[target_col].replace('Lost', np.nan)
train[target_col] = pd.to_numeric(train[target_col], errors='coerce')

print(f'Revenue missing: {train[target_col].isna().sum()} / {len(train)} ({train[target_col].isna().mean()*100:.1f}%)')

<h2 align=left style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
Feature Engineering
</font>
</h2>


In [ ]:
all_data = pd.concat([train, test], axis=0, ignore_index=True)

drop_cols = ['transaction_id', target_col, target_cls]
cat_cols_raw = all_data.select_dtypes(include='object').columns.tolist()
cat_cols_raw = [c for c in cat_cols_raw if c not in drop_cols]

for col in ['holiday_season', 'marketing_campaign']:
    if col in all_data.columns:
        all_data[col] = all_data[col].astype(int)

all_data['customer_ratio'] = all_data['new_customers'] / (all_data['new_customers'] + all_data['returning_customers']).replace(0, np.nan)
all_data['online_ratio'] = all_data['online_players'] / all_data['estimated_active_players'].replace(0, np.nan)
all_data['story_ratio'] = all_data['story_mode_players'] / all_data['estimated_active_players'].replace(0, np.nan)
all_data['gta_online_ratio'] = all_data['gta_online_players'] / all_data['online_players'].replace(0, np.nan)
all_data['peak_per_active'] = all_data['peak_concurrent_players'] / all_data['estimated_active_players'].replace(0, np.nan)
all_data['units_per_active'] = all_data['units_sold'] / all_data['estimated_active_players'].replace(0, np.nan)
all_data['market_gdp_interaction'] = all_data['gaming_market_size'] * all_data['gdp_per_capita_usd']
all_data['population_market'] = all_data['population_millions'] * all_data['gaming_market_size']
all_data['internet_gdp'] = all_data['internet_penetration_percentage'] * all_data['gdp_per_capita_usd']
all_data['playtime_session'] = all_data['average_playtime_hours'] * all_data['average_session_length_minutes']
all_data['refund_impact'] = all_data['refund_rate_percentage'] * all_data['units_sold']
all_data['total_players'] = all_data['online_players'] + all_data['story_mode_players']
all_data['weekend_weekday_ratio'] = all_data['weekend_sales_percentage'] / all_data['weekday_sales_percentage'].replace(0, np.nan)

label_encoders = {}
for col in cat_cols_raw:
    le = LabelEncoder()
    all_data[col] = le.fit_transform(all_data[col].astype(str))
    label_encoders[col] = le

train_processed = all_data.iloc[:len(train)].copy()
test_processed = all_data.iloc[len(train):].copy()

feature_cols = [c for c in all_data.columns if c not in ['transaction_id', target_col, target_cls, 'is_revenue_missing']]

train_features = train_processed[feature_cols].values
test_features = test_processed[feature_cols].values
train_features = np.nan_to_num(train_features, nan=-1.0).astype(np.float32)
test_features = np.nan_to_num(test_features, nan=-1.0).astype(np.float32)

y_cls = np.array(train_processed[target_cls].astype(str).values).flatten()
y_reg = np.array(train_processed[target_col].values, dtype=float)
has_revenue = ~np.isnan(y_reg)

print(f'Features shape: {train_features.shape}')
print(f'Train rows with revenue: {has_revenue.sum()} / {len(has_revenue)}')

<h2 align="left" style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
Modeling
</font>
</h2>

<p dir="ltr" style="direction: ltr; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size="3">
    Now that you have cleaned the data and possibly added or removed features from it, it is time to train a model that can predict the target variable of this problem.
</font>
</p>

In [ ]:
N_SPLITS_CLS = 5
cls_oof = np.zeros((len(train_features), 2))
cls_test_preds = np.zeros((len(test_features), 2))
skf_cls = StratifiedKFold(n_splits=N_SPLITS_CLS, shuffle=True, random_state=42)

for fold, (tr_idx, val_idx) in enumerate(skf_cls.split(train_features, y_cls)):
    print(f'Classification fold {fold+1}/{N_SPLITS_CLS}...')
    X_tr, X_val = train_features[tr_idx], train_features[val_idx]
    y_tr, y_val = y_cls[tr_idx], y_cls[val_idx]

    cat_cls = CatBoostClassifier(
        iterations=2000, learning_rate=0.03, depth=8,
        l2_leaf_reg=3, min_data_in_leaf=10,
        random_seed=42 + fold, verbose=0,
        early_stopping_rounds=100, eval_metric='Logloss'
    )
    cat_cls.fit(X_tr, y_tr, eval_set=(X_val, y_val), verbose=0)

    xgb_cls = xgb.XGBClassifier(
        n_estimators=2000, learning_rate=0.03, max_depth=8,
        subsample=0.8, colsample_bytree=0.8, reg_lambda=3,
        random_state=42 + fold, eval_metric='logloss',
        early_stopping_rounds=100, use_label_encoder=False, verbosity=0
    )
    xgb_cls.fit(X_tr, (y_tr == 'Physical').astype(int),
                eval_set=[(X_val, (y_val == 'Physical').astype(int))], verbose=False)

    lgb_cls = lgb.LGBMClassifier(
        n_estimators=2000, learning_rate=0.03, max_depth=8,
        subsample=0.8, colsample_bytree=0.8, reg_lambda=3,
        random_state=42 + fold, verbose=-1
    )
    lgb_cls.fit(X_tr, (y_tr == 'Physical').astype(int),
                eval_set=[(X_val, (y_val == 'Physical').astype(int))],
                callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)])

    ensemble_val = 0.4 * cat_cls.predict_proba(X_val) + 0.3 * xgb_cls.predict_proba(X_val) + 0.3 * lgb_cls.predict_proba(X_val)
    cls_oof[val_idx] = ensemble_val

    ensemble_test = 0.4 * cat_cls.predict_proba(test_features) + 0.3 * xgb_cls.predict_proba(test_features) + 0.3 * lgb_cls.predict_proba(test_features)
    cls_test_preds += ensemble_test / N_SPLITS_CLS

    fold_ll = log_loss(y_val, ensemble_val)
    print(f'  Fold {fold+1} LogLoss: {fold_ll:.5f}')

oof_ll = log_loss(y_cls, cls_oof)
print(f'\nOverall OOF LogLoss: {oof_ll:.5f}')

In [ ]:
N_SPLITS_REG = 5
X_reg_train = train_features[has_revenue]
y_reg_train = y_reg[has_revenue]
y_reg_log = np.log1p(y_reg_train)

reg_oof = np.zeros(len(X_reg_train))
reg_test_preds = np.zeros(len(test_features))
kf_reg = KFold(n_splits=N_SPLITS_REG, shuffle=True, random_state=42)

for fold, (tr_idx, val_idx) in enumerate(kf_reg.split(X_reg_train, y_reg_train)):
    print(f'Regression fold {fold+1}/{N_SPLITS_REG}...')
    X_tr, X_val = X_reg_train[tr_idx], X_reg_train[val_idx]
    y_tr_log, y_val_log = y_reg_log[tr_idx], y_reg_log[val_idx]
    y_val_actual = y_reg_train[val_idx]

    cat_reg = CatBoostRegressor(
        iterations=2000, learning_rate=0.03, depth=8,
        l2_leaf_reg=3, min_data_in_leaf=10,
        random_seed=42 + fold, verbose=0,
        early_stopping_rounds=100, eval_metric='RMSE'
    )
    cat_reg.fit(X_tr, y_tr_log, eval_set=(X_val, y_val_log), verbose=0)

    xgb_reg = xgb.XGBRegressor(
        n_estimators=2000, learning_rate=0.03, max_depth=8,
        subsample=0.8, colsample_bytree=0.8, reg_lambda=3,
        random_state=42 + fold, early_stopping_rounds=100, verbosity=0
    )
    xgb_reg.fit(X_tr, y_tr_log, eval_set=[(X_val, y_val_log)], verbose=False)

    lgb_reg = lgb.LGBMRegressor(
        n_estimators=2000, learning_rate=0.03, max_depth=8,
        subsample=0.8, colsample_bytree=0.8, reg_lambda=3,
        random_state=42 + fold, verbose=-1
    )
    lgb_reg.fit(X_tr, y_tr_log, eval_set=[(X_val, y_val_log)],
                callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)])

    ensemble_val_log = 0.4 * cat_reg.predict(X_val) + 0.3 * xgb_reg.predict(X_val) + 0.3 * lgb_reg.predict(X_val)
    ensemble_val_actual = np.expm1(ensemble_val_log)
    reg_oof[val_idx] = ensemble_val_actual

    ensemble_test_log = 0.4 * cat_reg.predict(test_features) + 0.3 * xgb_reg.predict(test_features) + 0.3 * lgb_reg.predict(test_features)
    reg_test_preds += np.expm1(ensemble_test_log) / N_SPLITS_REG

    fold_rmsle = np.sqrt(mean_squared_log_error(y_val_actual, ensemble_val_actual))
    print(f'  Fold {fold+1} RMSLE: {fold_rmsle:.5f}')

oof_rmsle = np.sqrt(mean_squared_log_error(y_reg_train, reg_oof))
print(f'\nOverall OOF RMSLE: {oof_rmsle:.5f}')

<h3 align="left" style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
Evaluation Metric
</font>
</h3>

<p dir="ltr" style="direction: ltr; text-align: justify; line-height:200%;font-family:vazir;font-size:medium">
<font face="vazir" size="3">
    To evaluate the performance of your model before submitting the answer, you can use the <code>log_loss</code> (for the sales channel classification section) and <code>mean_squared_log_error</code> (for calculating RMSLE in the revenue prediction section) metrics from the <code>scikit-learn</code> library. 
    <br><br>
    It is recommended to simulate your score using the judging system's formula.
</font>
</p>

In [ ]:
total_score = 100 * np.exp(-1.3 * (0.35 * oof_ll + 0.65 * oof_rmsle))
print(f'LogLoss: {oof_ll:.5f}')
print(f'RMSLE: {oof_rmsle:.5f}')
print(f'Simulated Score: {total_score:.2f} / 100')

<h2 align="left" style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
 Prediction for Test Data and Output
</font>
</h2>

<p dir="ltr" style="direction: ltr;text-align: left;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazir" size="3">
    After feature engineering and modeling, you have an algorithm that can take you from independent variables to target variables.
    <br>
    Use this model to predict the samples present in the test data and prepare the results in the format of the table (<code>dataframe</code>) below.
</font>
</p>

<div dir="ltr" style="direction: ltr;text-align: left;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazir" size="3">
    
| Column | Description |
|:------:|:---:|
|transaction_id|Unique transaction identifier (exactly the same as the test data)|
|gross_revenue_usd|Your prediction for the revenue amount in USD (decimal number)|
|sales_channel|Your prediction for the sales channel (Physical or Digital)|
    
</font>
</div>

In [ ]:
test_pred_cls = ['Digital' if p[0] > p[1] else 'Physical' for p in cls_test_preds]
test_pred_rev = np.maximum(reg_test_preds, 0)

submission = pd.DataFrame({
    'transaction_id': test['transaction_id'],
    'sales_channel_pred': test_pred_cls,
    'revenue_pred': test_pred_rev
})
print(f'Submission shape: {submission.shape}')
submission.head(10)

<h2 align="left" style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
<b>Result Generator Cell</b>
</font>
</h2>

<p dir="ltr" style="direction: ltr; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size="3">
    Run the cell below to generate the <code>result.zip</code> file.
    <br>
    You must upload the <code>result.zip</code> file in Quera and wait for the judging result!
    <br>
    This file includes <code>submission.csv</code>, this very notebook you are coding in, and your saved model.
</font>
</p>

In [ ]:
import zipfile

def compress(file_names):
    print("File Paths:")
    print(file_names)
    compression = zipfile.ZIP_DEFLATED
    with zipfile.ZipFile("result.zip", mode="w") as zf:
        for file_name in file_names:
            zf.write('./' + file_name, file_name, compress_type=compression)


submission.to_csv('submission.csv', index=False)

file_names = ['submission.csv', 'GTA.ipynb']
compress(file_names)